# Online dependence estimation — experiment analysis

Loads the `experiment_<scenario>.csv` files produced by the C++ harness and plots each
estimator's tracking of the true Spearman's rho, plus an RMSE summary.

**Prerequisites** (run in a terminal from the repo root first):
```bash
pip install pandas matplotlib numpy jupyter
make experiment
for s in constant drift jump stochastic oscillate; do ./experiment $s; done
```
Then run this notebook top to bottom.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run experiments from the repo root, so CSVs land there. Adjust if needed.
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
SCENARIOS = ['constant', 'drift', 'jump', 'stochastic', 'oscillate']
BURN_IN = 500  # steps to skip before scoring, matches the C++ harness

# Consistent colours across scenarios (match the C++/React client).
COLORS = {
    'xiao': '#ffb800',
    'xiao_ew': '#ff5555',
    'hermite': '#22d3ee',
    'hermite_ew': '#a78bfa',
    'windowed_rank': '#888888',
}
print('looking for CSVs in:', DATA_DIR)

In [ ]:
def load(scenario):
    path = os.path.join(DATA_DIR, f'experiment_{scenario}.csv')
    return pd.read_csv(path)

def estimator_cols(df):
    return [c for c in df.columns if c not in ('t', 'true_rho_s')]

def plot_scenario(scenario):
    df = load(scenario)
    plt.figure(figsize=(11, 4.5))
    plt.plot(df['t'], df['true_rho_s'], color='black', lw=2.4, label='true rho_S')
    for col in estimator_cols(df):
        plt.plot(df['t'], df[col], lw=1.1, alpha=0.85,
                 color=COLORS.get(col), label=col)
    plt.axvline(BURN_IN, color='grey', ls=':', lw=1, label=f'burn-in ({BURN_IN})')
    plt.xlabel('t'); plt.ylabel('Spearman rho'); plt.ylim(-1, 1)
    plt.title(f'Online dependence tracking — {scenario}')
    plt.legend(loc='best', fontsize=8, ncol=2); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def rmse_table(scenario):
    df = load(scenario)
    tail = df[df['t'] >= BURN_IN]
    err = {c: float(np.sqrt(np.mean((tail[c] - tail['true_rho_s']) ** 2)))
           for c in estimator_cols(df)}
    return pd.Series(err, name=scenario)

In [ ]:
# Tracking plots, one per scenario.
for s in SCENARIOS:
    try:
        plot_scenario(s)
    except FileNotFoundError:
        print(f'missing experiment_{s}.csv — run ./experiment {s} first')

In [ ]:
# RMSE vs true Spearman (after burn-in): rows = estimators, cols = scenarios.
rows = []
for s in SCENARIOS:
    try:
        rows.append(rmse_table(s))
    except FileNotFoundError:
        print(f'missing experiment_{s}.csv — skipping')
summary = pd.concat(rows, axis=1) if rows else pd.DataFrame()
summary.style.format('{:.4f}').background_gradient(cmap='RdYlGn_r', axis=None) if not summary.empty else summary